### REad BTC Data

In [1]:
import pandas as pd

df = pd.read_csv("BTCUSDT.csv", sep='|')
df = df.rename(columns={'1502942400': 'timestamp', '4261.48': 'open', '4261.48.1': 'high', '4261.48.2': 'low', '4261.48.3': 'close', '1.775183': 'btc_volume_sell', '320.390851': 'usd_volume_buy', '0.075183': 'btc_volume_buy', '7564.906851': 'usd_volume_sell', '3': 'activity'}) 

df

,timestamp,open,high,low,close,btc_volume_sell,usd_volume_buy,btc_volume_buy,usd_volume_sell,activity
0,1502942460,4261.48,4261.48,4261.48,4261.48,0.000000,0.000000,0.000000,0.000000e+00,0
1,1502942520,4280.56,4280.56,4280.56,4280.56,0.261074,1117.542921,0.261074,1.117543e+03,2
2,1502942580,4261.48,4261.48,4261.48,4261.48,0.012008,51.171852,0.012008,5.117185e+01,3
3,1502942640,4261.48,4261.48,4261.48,4261.48,0.140796,599.999338,0.140796,5.999993e+02,1
4,1502942700,4261.48,4261.48,4261.48,4261.48,0.000000,0.000000,0.000000,0.000000e+00,0
...,...,...,...,...,...,...,...,...,...,...
3997969,1743334260,83149.78,83184.65,83132.43,83184.65,3.664100,236173.309690,2.840050,3.047000e+05,1811
3997970,1743334320,83184.65,83184.66,83141.85,83141.85,2.287810,60844.870800,0.731660,1.902545e+05,1300
3997971,1743334380,83141.85,83141.86,83100.00,83104.94,3.499020,164623.561700,1.980910,2.908151e+05,1411
3997972,1743334440,83106.00,83110.76,83038.02,83055.34,16.144520,231423.308990,2.786300,1.341206e+06,4368


### Derive volatility

In [2]:
import numpy as np

# volatility
df['datetime'] = pd.to_datetime(df['timestamp'], unit='s')
df.set_index('datetime', inplace=True)
df['log_return'] = np.log(df['close'] / df['close'].shift(1))
window_size = 30 * 60 * 60
df['30d_volatility'] = df['log_return'].rolling(window=window_size).std() * np.sqrt(window_size)
df[['close', 'log_return', '30d_volatility']].tail(10)

# avg volume, avg buy volume, avg sell volume
df['datetime'] = pd.to_datetime(df['timestamp'], unit='s')
df.set_index('datetime', inplace=True)
df['usd_total_volume'] = df['usd_volume_buy'] + df['usd_volume_sell']
df['7d_usd_volume_buy_avg'] = df['usd_volume_buy'].rolling(window=7*24*60, center=False).mean()
df['7d_usd_volume_sell_avg'] = df['usd_volume_sell'].rolling(window=7*24*60, center=False).mean()
df['7d_usd_volume_avg'] = df['usd_total_volume'].rolling(window=7*24*60, center=False).mean()

max_vol = max(df[df['30d_volatility'].notna()]['30d_volatility'])
print(max_vol)

window = 4 * 60
#first value most important, last
df['forward_usd_volume_buy_avg'] =  df['usd_volume_buy'].ewm(span=window, adjust=False).mean()
df['forward_usd_volume_sell_avg'] =  df['usd_volume_sell'].ewm(span=window, adjust=False).mean()
df['forward_usd_volume_avg'] =  df['usd_total_volume'].ewm(span=window, adjust=False).mean()

df['price_change_4h'] = (df['close'].shift(-15) - df['close'].shift(window) ) / df['close']
df['price_change_4h_tanh'] = np.tanh(df['price_change_4h'])
df['price_change_4h_volatility_tanh'] = np.tanh(df['price_change_4h'] * (max_vol - df['30d_volatility']))

df['price_change_4h_ema'] = df['price_change_4h_tanh'].ewm(span=60, adjust=False).mean()
df['price_change_4h_volatility_ema'] = df['price_change_4h_volatility_tanh'].ewm(span=60, adjust=False).mean()

df[['close','usd_volume_buy', 'forward_usd_volume_buy_avg', '7d_usd_volume_buy_avg', 'price_change_4h']]

0.9900359831658481


,close,usd_volume_buy,forward_usd_volume_buy_avg,7d_usd_volume_buy_avg,price_change_4h
datetime,,,,,
2017-08-17 04:01:00,4261.48,0.000000,0.000000,NaN,NaN
2017-08-17 04:02:00,4280.56,1117.542921,9.274215,NaN,NaN
2017-08-17 04:03:00,4261.48,51.171852,9.621913,NaN,NaN
2017-08-17 04:04:00,4261.48,599.999338,14.521311,NaN,NaN
2017-08-17 04:05:00,4261.48,0.000000,14.400802,NaN,NaN
...,...,...,...,...,...
2025-03-30 11:31:00,83184.65,236173.309690,216449.362461,534779.603856,NaN
2025-03-30 11:32:00,83141.85,60844.870800,215158.038878,534767.281993,NaN
2025-03-30 11:33:00,83104.94,164623.561700,214738.665624,534748.305245,NaN


In [3]:
from math import sqrt
import numpy as np
import pandas as pd

df['raw_volume_surge'] = (
    df['forward_usd_volume_buy_avg'] / df['7d_usd_volume_buy_avg'] -
    df['forward_usd_volume_sell_avg'] / df['7d_usd_volume_sell_avg']
)

# adjust by volatility
max_vol = max(df[df['30d_volatility'].notna()]['30d_volatility'])
print(max_vol)

df['adjusted_surge'] = np.sqrt(df['raw_volume_surge'] * (max_vol - df['30d_volatility'] + 1e-8) )
df['adjusted_surge_2'] = np.sqrt((df['raw_volume_surge'] * (max_vol - df['30d_volatility'] + 1e-8)) * abs(df['price_change_4h']))

df['adjusted_surge'] = (df['raw_volume_surge'] * (max_vol - df['30d_volatility'] + 1e-8) )
df['adjusted_surge_2'] = ((df['raw_volume_surge'] * (max_vol - df['30d_volatility'] + 1e-8)) * abs(df['price_change_4h']))

df['volume_surge'] = np.tanh(df['raw_volume_surge'])
df['volume_surge_volatility'] = np.tanh(df['adjusted_surge'])
df['volume_surge_price'] = np.tanh(df['adjusted_surge_2'])


dff = df.copy()
dff = dff[dff['volume_surge'].notna()]
dff = dff[dff['volume_surge_volatility'].notna()]
print(len(dff))
dff[['timestamp', 'volume_surge', 'volume_surge_volatility', 'volume_surge_price', 'price_change_4h']].sort_values(by='volume_surge_price',ascending= True).head(20)


0.9900359831658481


/home/kosmo/.local/lib/python3.10/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in sqrt
  result = getattr(ufunc, method)(*inputs, **kwargs)
/home/kosmo/.local/lib/python3.10/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in sqrt
  result = getattr(ufunc, method)(*inputs, **kwargs)


3889974


,timestamp,volume_surge,volume_surge_volatility,volume_surge_price,price_change_4h
datetime,,,,,
2019-09-24 19:29:00,1569353340,-0.841425,-0.671569,-0.159339,-0.197529
2019-09-24 19:32:00,1569353520,-0.845473,-0.676619,-0.144121,-0.176376
2019-09-24 19:45:00,1569354300,-0.887913,-0.731368,-0.143082,-0.154638
2019-09-24 19:44:00,1569354240,-0.898435,-0.748168,-0.141256,-0.146790
2019-09-24 19:33:00,1569353580,-0.843921,-0.674666,-0.140776,-0.172982
2019-09-24 19:28:00,1569353280,-0.831580,-0.659486,-0.139402,-0.177188
2020-03-12 10:46:00,1584009960,-0.592384,-0.454312,-0.137526,-0.282385
2019-09-24 19:30:00,1569353400,-0.849472,-0.681724,-0.136356,-0.164852
2019-09-24 19:31:00,1569353460,-0.844828,-0.675821,-0.134303,-0.164503


In [4]:
# window_4h = int(4 * 60 / 2)  # 2 hours before and after

# max_vol = max(df[df['30d_volatility'].notna()]['30d_volatility'])
# print(max_vol)

# # momentum change (as percentage)
# df['momentum_change'] = (df['close'].shift(-window_4h) - df['close'].shift(window_4h)) / df['close'].shift(window_4h)
# df['momentum_change_volatility'] = ((df['close'].shift(-window_4h) - df['close'].shift(window_4h)) / df['close'].shift(window_4h)) * (max_vol - df['30d_volatility'] + 1e-8) 

# # smooth this using an EMA to reduce noise
# df['momentum_ema'] = df['momentum_change'].ewm(span=60, adjust=False).mean()
# df['momentum_volatility_ema'] = df['momentum_change_volatility'].ewm(span=60, adjust=False).mean()

# # Inspect the results
# df[['close', 'momentum_change', 'momentum_change_volatility', 'momentum_volatility_ema', 'momentum_ema']].sort_values(by='momentum_volatility_ema',ascending= False).head(20)

### connect with headlines

In [5]:
df_news = pd.read_csv("ok.csv")
df_news.rename(columns={'timestamp': 'newsTimestamp'}, inplace=True)
df_news.rename(columns={'from': 'source'}, inplace=True)
df_news.rename(columns={'text': 'title'}, inplace=True)
df_news.rename(columns={'date_unixtime': 'timestamp'}, inplace=True)
df_news.rename(columns={'date': 'datetime'}, inplace=True)
df_news = df_news.drop('Unnamed: 0',axis='columns')
df_news['newsTimestamp'] = pd.to_datetime(df_news['datetime']).astype(int) // 10**9
df['timestamp'] = df['timestamp'].astype(int)
df = df.sort_values(by='timestamp')

df_news = df_news[df_news['title'].str.len() > 3]

news_times = df_news['newsTimestamp'].values
df_timestamps = df['timestamp'].values

# Find the closest previous timestamp index (open price before news)
news_indices = np.searchsorted(df_timestamps, news_times, side='right') - 1
news_indices = np.clip(news_indices, 0, len(df) - 1)  # Ensure valid indices

df_news['open_before_news'] = df.iloc[news_indices]['open'].values
df_news['30d_volatility'] = df.iloc[news_indices]['30d_volatility'].values
df_news['volume_surge'] = df.iloc[news_indices]['volume_surge'].values
df_news['volume_surge_volatility'] = df.iloc[news_indices]['volume_surge_volatility'].values
df_news['volume_surge_price'] = df.iloc[news_indices]['volume_surge_price'].values
# df_news['momentum_change'] = df.iloc[news_indices]['momentum_change'].values
# df_news['momentum_change_volatility'] = df.iloc[news_indices]['momentum_change_volatility'].values
# df_news['momentum_ema'] = df.iloc[news_indices]['momentum_ema'].values
# df_news['momentum_volatility_ema'] = df.iloc[news_indices]['momentum_volatility_ema'].values
df_news['price_change_4h'] = df.iloc[news_indices]['price_change_4h'].values
df_news['price_change_4h_tanh'] = df.iloc[news_indices]['price_change_4h_tanh'].values
df_news['price_change_4h_volatility_tanh'] = df.iloc[news_indices]['price_change_4h_volatility_tanh'].values
df_news['price_change_4h_ema'] = df.iloc[news_indices]['price_change_4h_ema'].values
df_news['price_change_4h_volatility_ema'] = df.iloc[news_indices]['price_change_4h_volatility_ema'].values



print(len(df_news))
print(df_news.columns)

df_news.sort_values(by='price_change_4h',ascending= True).head(10)

20278
Index(['datetime', 'timestamp', 'title', 'source', 'newsTimestamp',
       'open_before_news', '30d_volatility', 'volume_surge',
       'volume_surge_volatility', 'volume_surge_price', 'price_change_4h',
       'price_change_4h_tanh', 'price_change_4h_volatility_tanh',
       'price_change_4h_ema', 'price_change_4h_volatility_ema'],
      dtype='object')


,datetime,timestamp,title,source,newsTimestamp,open_before_news,30d_volatility,volume_surge,volume_surge_volatility,volume_surge_price,price_change_4h,price_change_4h_tanh,price_change_4h_volatility_tanh,price_change_4h_ema,price_change_4h_volatility_ema
10389,2020-03-12T13:28:15,1584012495,Cost of liquidity spiking,unfolded.,1584019695,5990.00,0.326517,-0.258797,-0.173926,-0.045782,-0.260731,-0.254979,-0.171294,-0.228703,-0.153522
10383,2020-03-12T12:30:00,1584009000,Tokyo police arrest two for taking possession ...,unfolded.,1584016200,6085.76,0.325006,-0.388172,-0.265881,-0.064314,-0.236406,-0.232098,-0.155934,-0.205916,-0.138898
10384,2020-03-12T12:35:27,1584009327,Another leg up for implied volatility,unfolded.,1584016527,6119.76,0.325041,-0.379887,-0.259853,-0.062583,-0.235626,-0.231360,-0.155420,-0.210323,-0.141779
10385,2020-03-12T12:54:14,1584010454,new ฿ daily low: \n\nBitstamp - $5965\nBitfine...,unfolded.,1584017654,5967.72,0.325572,-0.375274,-0.256317,-0.061072,-0.233243,-0.229104,-0.153753,-0.216731,-0.145777
10382,2020-03-12T11:44:49,1584006289,LocalBitcoins seller charged with money launde...,unfolded.,1584013489,5926.26,0.318579,-0.599217,-0.433824,-0.107187,-0.231598,-0.227545,-0.154267,-0.200906,-0.137782
10381,2020-03-12T11:44:49,1584006289,CME Group to close Chicago trading floor as a ...,unfolded.,1584013489,5926.26,0.318579,-0.599217,-0.433824,-0.107187,-0.231598,-0.227545,-0.154267,-0.200906,-0.137782
10388,2020-03-12T13:13:06,1584011586,3rd biggest daily liquidations the last 2 year...,unfolded.,1584018786,6027.92,0.326381,-0.284981,-0.192098,-0.044395,-0.228383,-0.224494,-0.150418,-0.219937,-0.147670
10386,2020-03-12T12:58:20,1584010700,Whopping $520m liquidated so far on BitMEX and...,unfolded.,1584017900,6068.84,0.326137,-0.343383,-0.233251,-0.053154,-0.223902,-0.220234,-0.147563,-0.217591,-0.146292
10387,2020-03-12T13:03:58,1584011038,BitMEX OI $670m,unfolded.,1584018238,6061.27,0.326275,-0.315866,-0.213735,-0.047821,-0.220457,-0.216954,-0.145295,-0.217733,-0.146305
10380,2020-03-12T11:19:52,1584004792,(ERC20) balances on exchanges by +125% within ...,unfolded.,1584011992,6354.75,0.315167,-0.584309,-0.423110,-0.086952,-0.193083,-0.190719,-0.129574,-0.164060,-0.114164


In [6]:
# import re
# import numpy as np
# from sklearn.metrics import classification_report, r2_score
# from sklearn.metrics import mean_squared_error
# from sklearn.model_selection import train_test_split
# from sklearn.feature_extraction.text import TfidfVectorizer
# from nltk.corpus import stopwords
# from nltk.tokenize import word_tokenize
# from nltk.stem import WordNetLemmatizer
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.svm import SVR

# df_news = df_news[df_news['price_change_4h'].notna()]
# df_news = df_news[df_news['volume_surge_volatility'].notna()]

# stop_words = set(stopwords.words('english'))
# lemmatizer = WordNetLemmatizer()

# def clean_text(text):
#     text = text.lower()
#     text = re.sub(r'\d+', '', text)  # Remove numbers
#     text = re.sub(r'[^\w\s]', '', text)  # Remove punctuation
#     words = word_tokenize(text)
#     words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
#     #print(' '.join(words))
#     return ' '.join(words)

# weights = ['volume_surge', 'volume_surge_volatility', 'volume_surge_price', 
#             'price_change_4h_tanh', 'price_change_4h_volatility_tanh' ,
#               'price_change_4h_ema', 'price_change_4h_volatility_ema']

# column_for_correlation = 'price_change_4h'
# cleaned = df_news['title'].apply(clean_text)
# vectorizer =  TfidfVectorizer(max_df=0.05, min_df=1,ngram_range=(1,7),sublinear_tf=False)
# x = vectorizer.fit_transform(cleaned)

# for w in weights:    
#     y = df_news[w]
#     x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=13)

#     svm_clf = SVR(kernel='linear', C=0.1, epsilon=0.5, gamma='scale')  
#     svm_clf.fit(x_train, y_train)
#     y_pred = svm_clf.predict(x_test)
#     mse = mean_squared_error(y_test, y_pred)
#     print(w)
#     print(f"Mean Squared Error (MSE): {mse}")
#     rmse = np.sqrt(mse)
#     print(f"Root Mean Squared Error (RMSE): {rmse}")
#     r2 = r2_score(y_test, y_pred)
#     print(f"R² Score: {r2}")
#     print('')

In [7]:
from sklearn.preprocessing import MinMaxScaler


selected_columns = ['timestamp', 'title', 'source', 'volume_surge_price', 'price_change_4h']
df_save = df_news[selected_columns]
df_save = df_save.drop(df_save.nlargest(10, 'volume_surge_price').index)
df_save = df_save.drop(df_save.nsmallest(10, 'volume_surge_price').index)


df_save.sort_values('volume_surge_price', ascending=True).head(30)

neg_mask = df_save['volume_surge_price'] < 0
pos_mask = df_save['volume_surge_price'] > 0

scaler_neg = MinMaxScaler(feature_range=(-1, 0))
scaler_pos = MinMaxScaler(feature_range=(0, 1))

df_save.loc[neg_mask, 'volume_surge_price'] = scaler_neg.fit_transform(df_save.loc[neg_mask, ['volume_surge_price']])
df_save.loc[pos_mask, 'volume_surge_price'] = scaler_pos.fit_transform(df_save.loc[pos_mask, ['volume_surge_price']])


#df_save.sort_values('volume_surge_price', ascending=False).head(30)
df_save.to_csv('output.csv', index=False)

In [8]:
import re
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.discriminant_analysis import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.stats import pearsonr

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVR

df_news = df_news[df_news['price_change_4h'].notna()]
df_news = df_news[df_news['volume_surge_volatility'].notna()]

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = text.lower()
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    words = word_tokenize(text)
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    return ' '.join(words)

cleaned_titles = df_news['title'].apply(clean_text)
vectorizer = TfidfVectorizer(max_df=0.1, min_df=1, ngram_range=(1,7), sublinear_tf=False)
X = vectorizer.fit_transform(cleaned_titles)

weights = ['volume_surge', 'volume_surge_volatility', 'volume_surge_price',
            'price_change_4h_tanh', 'price_change_4h_volatility_tanh' , 'price_change_4h_ema', 'price_change_4h_volatility_ema']
column_for_correlation = 'price_change_4h'

results = []

# Loop through each target
for w in weights:


    df_copy = df_news.copy()

    col = w
    df_copy = df_copy.drop(df_copy.nlargest(3, col).index)
    df_copy = df_copy.drop(df_copy.nsmallest(3, col).index)

    cleaned_titles = df_copy['title'].apply(clean_text)
    vectorizer = TfidfVectorizer(max_df=0.1, min_df=1, ngram_range=(1,7), sublinear_tf=False)
    X = vectorizer.fit_transform(cleaned_titles)

    neg_mask = df_copy[col] < 0
    pos_mask = df_copy[col] > 0
    neg_scaler = MinMaxScaler(feature_range=(-1, 0))
    pos_scaler = MinMaxScaler(feature_range=(0, 1))

    scaled_s = pd.Series(index=df_copy.index, dtype='float64')
    if neg_mask.any():
        scaled_s.loc[neg_mask.index[neg_mask]] = neg_scaler.fit_transform(
            df_copy.loc[neg_mask, col].values.reshape(-1, 1)
        ).flatten()
    if pos_mask.any():
        scaled_s.loc[pos_mask.index[pos_mask]] = pos_scaler.fit_transform(
            df_copy.loc[pos_mask, col].values.reshape(-1, 1)
        ).flatten()

    y = scaled_s
    
    
    #y = df_news[w]
    y_corr = df_copy[column_for_correlation]
    
    X_train, X_test, y_train, y_test, y_corr_train, y_corr_test = train_test_split(
        X, y, y_corr, test_size=0.2, random_state=13
    )
    
    model = SVR(kernel='poly', C=0.25, epsilon=0.001, degree=7, gamma='scale')
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    mape = np.mean(np.abs((y_test - y_pred) / y_test))
    smape = 100/len(y_test) * np.sum(2 * np.abs(y_pred - y_test) / (np.abs(y_pred) + np.abs(y_test)))
    r2 = r2_score(y_test, y_pred)

    nonzero_mask = y_test != 0
    if np.any(nonzero_mask):
        mape = np.mean(np.abs((y_test[nonzero_mask] - y_pred[nonzero_mask]) / y_test[nonzero_mask])) * 100
    else:
        mape = np.nan

    print(f'{w} | y_pred var: {np.var(y_pred)} | y_corr_test var: {np.var(y_corr_test)} | smape: {smape}')

    
    corr_with_price_change_4h, _ = pearsonr(y_pred, y_corr_test)

    results.append({
        'Metric': w,
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape,
        'SMAPE': smape,
        'R²': r2,
        f'Corr_with_{column_for_correlation}': corr_with_price_change_4h
    })

# Convert results to DataFrame
results_df = pd.DataFrame(results)
print(results_df)

# Plotting
plt.figure(figsize=(14, 6))
metrics_to_plot = ['RMSE', 'MAE']

# Bar plot for errors
results_df_melted = results_df.melt(id_vars='Metric', value_vars=metrics_to_plot)
sns.barplot(data=results_df_melted, x='Metric', y='value', hue='variable')
plt.xticks(rotation=45)
plt.title('Error Metrics by Model')
plt.ylabel('Error Value')
plt.grid(axis='y')
plt.show()

# Bar plot for R² and correlation
plt.figure(figsize=(14, 5))
sns.barplot(x='Metric', y='R²', data=results_df, color='skyblue', label='R²')
sns.barplot(x='Metric', y=f'Corr_with_{column_for_correlation}', data=results_df, color='salmon', label=f'Corr with {column_for_correlation}')
plt.xticks(rotation=45)
plt.title('R² and Correlation with price_change_4h')
plt.legend()
plt.grid(axis='y')
plt.show()



volume_surge | y_pred var: 0.00033743461892663456 | y_corr_test var: 0.0003253470558134419 | smape: 173.8342102882551
volume_surge_volatility | y_pred var: 0.000300469777619468 | y_corr_test var: 0.00031438038509827744 | smape: 184.28522938327473
volume_surge_price | y_pred var: 4.11474566007068e-05 | y_corr_test var: 0.00032565548518349684 | smape: 154.14799024284434
price_change_4h_tanh | y_pred var: 0.0004837323049688781 | y_corr_test var: 0.00029145665492549426 | smape: 147.75103030235152
price_change_4h_volatility_tanh | y_pred var: 0.00046947229572413444 | y_corr_test var: 0.0003138442134536728 | smape: 147.84608263337395
price_change_4h_ema | y_pred var: 0.0004662724750118651 | y_corr_test var: 0.0002999674295846068 | smape: 147.63806015963564
price_change_4h_volatility_ema | y_pred var: 0.0006984275266482509 | y_corr_test var: 0.00032456373056807293 | smape: 147.08391804378303
                            Metric       MSE      RMSE       MAE  \
0                     volume_surge

In [9]:
cleaned_titles = df_news['title'].apply(clean_text)
vectorizer = TfidfVectorizer(max_df=0.1, min_df=1, ngram_range=(1,7), sublinear_tf=False)
X = vectorizer.fit_transform(cleaned_titles)

scaler = MinMaxScaler(feature_range=(-1, 1))
s_scaled = scaler.fit_transform(df_news['volume_surge_price'].values.reshape(-1, 1))
y = pd.Series(s_scaled.flatten(), index=df_news[w].index)

#y = df_news['price_change_4h_volatility_ema']

X_train, X_test, y_train, y_test, _, X_demo = train_test_split(
    X, y, df_news['title'],  test_size=0.2, random_state=13
)

model = SVR(kernel='poly', C=0.25, epsilon=0.001, degree=7, gamma='scale')
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

In [10]:
#_, X_demo, _, _ = train_test_split(df_news['title'], y, test_size=0.2, random_state=13)

df_demo = pd.DataFrame({"title": X_demo, "sentiment": y_test})

hello = df_demo.sort_values('sentiment', ascending=True).head(30)
for _, row in hello.iterrows():  
    print(f"{row['sentiment']} : {row['title']}")

-0.9397985320473834 : CME Group to close Chicago trading floor as a precaution due to coronavirus
-0.3373744794850798 : Whopping $520m liquidated so far on BitMEX and counting..
-0.07675822739856053 : New Bitcoin ETF proposal filed with SEC by Gold Fund Veteran
-0.07234917370919314 : Binance has added Shiba Inu/GBP trading pair.
-0.043212711540875914 : Fidelity-backed crypto security startup Fireblocks launches 'Secure Asset Transfer Network'
-0.00954753035569983 : European watchdogs demand detail on Facebook's cryptocurrency
0.017361611458127146 : There are now 11 blockchain unicorns in the world, with over $1B valuation
0.025875849996814998 : South Korea to launch 'Digital assets committee' in response to Terra implosion
0.026913049449967502 : Poloniex now allows fiat-to-crypto trading, credit card purchases
0.09656171014352977 : Major corporate NFT initiatives last year
0.12706261147237133 : $366 million liquidated from the cryptocurrency market in the past 60 minutes.
0.13347316506